# Capa 1 - Pipeline RAG (Retrieval-Augmented Generation)

## 1.Imports y configuración


In [1]:
# pip install --upgrade pymongo

In [2]:
#pip install faiss-cpu

In [3]:
import pandas as pd
import sys, os
from sentence_transformers import SentenceTransformer
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
import faiss

sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..')))

from data.corpus_canciones import get_collection
from src.rag_utils import chunking_por_estrofa, chunking_cancion_completa, mostrar_metricas_chunking, generar_o_cargar_embeddings,crear_indice_faiss, buscar_chunks_relevantes,cargar_modelo, generar_con_flan_t5, rag_completo, sin_rag
print("Funciona correctamente")

C:\PF-Chatbot-Musical\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Funciona correctamente


In [4]:
# Obtener la colección
collection = get_collection()

# Cargar
cursor = collection.find({}, {"_id": 0, "artista": 1, "nombre": 1, "letra": 1, "genero": 1, "anio":1, "titulo":1})
df_canciones = pd.DataFrame(list(cursor))

print(f"Total de canciones cargadas: {len(df_canciones)}")
display(df_canciones.head())

Total de canciones cargadas: 10384


,titulo,artista,genero,anio,letra
0,The Chain - 2004 Remaster,Fleetwood Mac,rock,1977,Fuck Listen to the wind blow Watch the sun ris...
1,Everlong,Foo Fighters,rock,1997,Hello I've waited here for you Everlong Tonigh...
2,"Lover, You Should've Come Over",Jeff Buckley,rock,1994,Looking out the door I see the rain Fall upon ...
3,Iris,Goo Goo Dolls,rock,2023,And I'd give up forever to touch you 'Cause I ...
4,Still Into You,Paramore,rock,2013,Can't count the years on one hand that we've b...


## 2. Chunking

### 2.1 Chunking por estrofa

In [5]:
# Convierte el DataFrame a una lista de diccionarios
datos_para_procesar = df_canciones.to_dict('records')

# Ejecucion de la función creada chunking_musical()
fragmentos_procesados_A = chunking_por_estrofa(datos_para_procesar)

### 2.2 Chunking por cancion completa

In [6]:
fragmentos_procesados_B = chunking_cancion_completa(datos_para_procesar)

###  2.3 Metricas Chunking por estrofa y Chunking por cancion completa

In [7]:
# Comparación de metricas
mostrar_metricas_chunking(fragmentos_procesados_A, "A (Por Estrofas)")
mostrar_metricas_chunking(fragmentos_procesados_B, "B (Canción Completa)")

--- MÉTRICAS ESTRATEGIA: A (Por Estrofas) ---
Total chunks: 24285
Tamaño promedio: 753 caracteres
Min/Max: 65/1492 caracteres
Ejemplo: 'Song: The Chain - 2004 Remaster | Artist: Fleetwood Mac | Genre: rock | Year: 1977
Lyrics: Fuck List...'
----------------------------------------

--- MÉTRICAS ESTRATEGIA: B (Canción Completa) ---
Total chunks: 10384
Tamaño promedio: 1647 caracteres
Min/Max: 150/51426 caracteres
Ejemplo: 'Song: The Chain - 2004 Remaster | Artist: Fleetwood Mac | Genre: rock | Year: 1977
Lyrics: Fuck List...'
----------------------------------------



## 3. Embeddings

#### Embeddings con la estrategia A: POR ESTROFA

In [8]:
# Cargamos el modelo multilingüe
print("Cargando modelo multilingüe...")
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

# Desempaqueta en DOS variables:
vectores_A, chunks_A = generar_o_cargar_embeddings(fragmentos_procesados_A, "emb_por_estrofa", modelo_emb)


print(f"Embeddings cargados: shape = {vectores_A.shape}")
print(f"Chunks con texto cargados: {len(chunks_A)}")

Cargando modelo multilingüe...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3937.27it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Cargando desde caché: emb_por_estrofa.pkl (Ahorrando tiempo...)
Embeddings cargados: shape = (24285, 384)
Chunks con texto cargados: 24285


#### Embeddings con la estrategia B: Canción Completa


In [9]:
# Cargamos el modelo multilingüe
print("Cargando modelo multilingüe...")
modelo_emb = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("\n--- ESTRATEGIA B: Cancion Completa ---")


# Desempaqueta en DOS variables:
vectores_B, chunks_B = generar_o_cargar_embeddings(fragmentos_procesados_B, "emb_cancion_completa", modelo_emb)

print(f"Embeddings cargados: shape = {vectores_B.shape}")

Cargando modelo multilingüe...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3781.46it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- ESTRATEGIA B: Cancion Completa ---
Cargando desde caché: emb_cancion_completa.pkl (Ahorrando tiempo...)
Embeddings cargados: shape = (10384, 384)


## 4. Creación indice FAISS

In [10]:
print("Construyendo Vector Store para Estrategia A...")
indice_A = crear_indice_faiss(vectores_A)

Construyendo Vector Store para Estrategia A...
Índice FAISS creado: 24285 vectores, dimensión 384


In [11]:
print("Construyendo Vector Store para Estrategia B...")
indice_B = crear_indice_faiss(vectores_B)

Construyendo Vector Store para Estrategia B...
Índice FAISS creado: 10384 vectores, dimensión 384


In [12]:
# Guardar índice FAISS para reutilizar en el chatbot
faiss.write_index(indice_A, "../notebooks/faiss_index_A.bin")
faiss.write_index(indice_B, "../notebooks/faiss_index_B.bin")
print("Índices FAISS guardados en disco.")

Índices FAISS guardados en disco.


## Busqueda Semantica

#### Chunks con estrategia A (POR ESTROFA)

In [13]:
pregunta = "songs about hate"

# Llamamos a la función pasando nuestras variables
mis_resultados = buscar_chunks_relevantes(
    pregunta=pregunta,
    indice_FAISS=indice_A,      # El índice FAISS
    chunks=fragmentos_procesados_A,  # chunking_por_estrofa
    modelo=modelo_emb,          # El modelo SentenceTransformer
    top_k=3                     # Los 3 mejores
)

Resultados para: 'songs about hate'

Resultado #1 (Similitud: 0.3430)
Song: My American Prayer | Artist: Downset | Genre: hip hop | Year: 1994
Lyrics: oubtless next in the death rate or love Suffocate beneath this fashion of hate I was taught to hate You and you weer taught to hate me love se...
--------------------------------------------------
Resultado #2 (Similitud: 0.3121)
Song: The Limit | Artist: Snowgoons,Viro The Virus | Genre: hip hop | Year: 2015
Lyrics: Human nature says the hate is in their blood Keep your friends close your enemies closer Y'all might find they're one and the same before it's...
--------------------------------------------------
Resultado #3 (Similitud: 0.3050)
Song: Golden Fleece | Artist: Hermit and the Recluse,Ka | Genre: hip hop | Year: 2018
Lyrics: ou that hate us is the half of you that need us I want compassion from the highest Food for the lowest Cures for the afflicted Rooves for the ...
--------------------------------------------------


#### Chunks con estrategia B (Cancion Completa)

In [14]:
pregunta = "songs about hate"

# Llamamos a la función pasando nuestras variables
mis_resultados = buscar_chunks_relevantes(
    pregunta=pregunta,
    indice_FAISS=indice_B,      # El índice FAISS
    chunks=fragmentos_procesados_A,  # chunking_por_estrofa
    modelo=modelo_emb,          # El modelo SentenceTransformer
    top_k=3                     # Queremos los 3 mejores
)

Resultados para: 'songs about hate'

Resultado #1 (Similitud: 0.2985)
Song: Last Night on Earth | Artist: Green Day | Genre: rock | Year: 2009
Lyrics: I text a postcard sent to you did it go through Sendin' all my love to you You are the moonlight of my life every night Givin' all my love to ...
--------------------------------------------------
Resultado #2 (Similitud: 0.2773)
Song: Driven Under | Artist: Seether | Genre: rock | Year: 2002
Lyrics: like she'd used it once before on him Then she told me she had a gun It sounded like she'd used it once before oh man We have to succumb to Th...
--------------------------------------------------
Resultado #3 (Similitud: 0.2769)
Song: The Stretch Armstrong and Bobbito Show on WKCR October 28 1993 | Artist: Nas,6'9,Jungle,Grand Wizard | Genre: hip hop | Year: 2014
Lyrics: lunt head Police Police want a nigga dead But I'm not goin' out like that black I kick the actual facts in solar Cold as a Polar Bear I swear ...
--------------------------

## 6. Generación de Respuestas

In [15]:
cargar_modelo()

Cargando google/flan-t5-base...


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1650.25it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Modelo Base listo.


In [16]:
# Prueba
contexto_prueba = "The song 'Black Hole Sun' by Soundgarden was released in 1994 and is a grunge classic."
pregunta_prueba = "When was Black Hole Sun released?"

# Llamamos a la función
respuesta = generar_con_flan_t5(contexto_prueba, pregunta_prueba)

print(f"Respuesta del modelo: {respuesta}")

Respuesta del modelo: 1994


## 7. Sistema RAG completo

In [17]:
print("EJECUTANDO RAG - ESTRATEGIA A (ESTROFAS)")
rag_completo(
    pregunta="¿Give me a song about loneliness?",
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

EJECUTANDO RAG - ESTRATEGIA A (ESTROFAS)

BUSCANDO: ¿Give me a song about loneliness?
Resultados para: '¿Give me a song about loneliness?'

Resultado #1 (Similitud: 0.4908)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: he vacant space The cried out tears and a never ending maze Oh I have found what only loneliness provides A strength within knowing I will fin...
--------------------------------------------------
Resultado #2 (Similitud: 0.4581)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: Oh yeah Loneliness is always looking for a friend It found me once And it has been around since then Loneliness is never waiting by the door I...
--------------------------------------------------
Resultado #3 (Similitud: 0.4527)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is bein

"met you Loneliness be over When will this loneliness be over? Life will flash before my eyes So scattered and lost I want to touch the other side And no one thinks they are to blame Why can't we see that when we bleed we bleed the same I can't get it right Get it right Since I met you"

In [18]:
print("EJECUTANDO RAG - ESTRATEGIA B (Cancion completa)")
rag_completo(
    pregunta="¿Give me a song about loneliness",
    indice_faiss=indice_B,
    chunks=fragmentos_procesados_B,
    modelo_emb=modelo_emb,
    top_k=3
)

EJECUTANDO RAG - ESTRATEGIA B (Cancion completa)

BUSCANDO: ¿Give me a song about loneliness
Resultados para: '¿Give me a song about loneliness'

Resultado #1 (Similitud: 0.4569)
Song: Lost Friends | Artist: Middle Kids | Genre: pop | Year: 2018
Lyrics: Lonely is the sound when the truth hits the ground I lost all my friends that day I lost all my friends We were sitting 'round and I remember ...
--------------------------------------------------
Resultado #2 (Similitud: 0.4471)
Song: Loneliness Knows Me By Name | Artist: Westlife | Genre: pop | Year: 2024
Lyrics: Oh yeah Loneliness is always looking for a friend It found me once And it has been around since then Loneliness is never waiting by the door I...
--------------------------------------------------
Resultado #3 (Similitud: 0.4423)
Song: Map of the Problematique | Artist: Muse | Genre: rock | Year: 2006
Lyrics: Fear and panic in the air I want to be free from desolation and despair And I feel like everything I sow Is being swep

"Loneliness knows me by name Loneliness knows everything I keep inside My endless thoughts in the silence of the night Loneliness is the one who made me see Ain't nobody else Who can make a change but me no Why why was I chosen?"

## Comparacion: RAG vs sin RAG

In [19]:
pregunta_prueba = "¿What is the most common feeling of the songs?"

# Sin RAG
print("--- PRUEBA SIN RAG ---")
print(sin_rag(pregunta_prueba))

# Con RAG
print("\n--- PRUEBA CON RAG ---")
# Aquí usas tu función estrella
rag_completo(
    pregunta=pregunta_prueba,
    indice_faiss=indice_A,
    chunks=fragmentos_procesados_A,
    modelo_emb=modelo_emb,
    top_k=3
)

--- PRUEBA SIN RAG ---
happy

--- PRUEBA CON RAG ---

BUSCANDO: ¿What is the most common feeling of the songs?
Resultados para: '¿What is the most common feeling of the songs?'

Resultado #1 (Similitud: 0.3054)
Song: Make You Feel That Way | Artist: Blackalicious | Genre: hip hop | Year: 2002
Lyrics: u feel that way make you feel that way Make you feel that way You know it's like like the most greatest feeling you could ever feel y'know Lik...
--------------------------------------------------
Resultado #2 (Similitud: 0.2747)
Song: The Feeling | Artist: Justin Bieber,Halsey | Genre: hip hop | Year: 2015
Lyrics:  you Or am I in love with the feeling...
--------------------------------------------------
Resultado #3 (Similitud: 0.2313)
Song: Nearly Witches (Ever Since We Met...) | Artist: Panic! at the Disco | Genre: rock | Year: 2011
Lyrics:  only shoot up with your perfume It's the only thing that makes me feel as good as you do Ever since we met I've got just one regret to live t...
-

'elation'

Sin RAG el modelo responde "happy" — una respuesta que pudo ser valida pero en el contexto del corpus (genero, letra etc) es posible que predomine otro sentimiento.
Con RAG recupera canciones reales del corpus y ancla la respuesta en ellas,demostrando el valor de la recuperación semántica. Elation = Júbilo/Euforia